# Image Seeds

Change the recorded seed. What stays fixed and what changes?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/stabilityai/sd-turbo

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors" "diffusers==0.30.3" "omegaconf" "wandb==0.31.0" "einops" "sentencepiece" "protobuf" "timm==1.0.22" "torchdiffeq"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os, random, tarfile, requests, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   CPU execution is slower; the same model experiments will run.")

IMG_DIR = "./generation-pets"
CACHE_DIR = "./cache"
OUT_DIR = "./lab5_outputs"
for d in [IMG_DIR, CACHE_DIR, OUT_DIR, f"{IMG_DIR}/dog", f"{IMG_DIR}/cat"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
def show_image_grid(images, titles=None, cols=4, figsize=(14, 7), suptitle=None):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(images):
            img = images[i]
            if isinstance(img, (str, Path)):
                img = Image.open(img).convert("RGB")
            ax.imshow(img)
            if titles is not None:
                ax.set_title(titles[i], fontsize=11, fontweight="bold")
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

n_show = min(8, len(image_paths))
show_image_grid(
    image_paths[:n_show],
    titles=[y.upper() for y in image_labels[:n_show]],
    cols=4,
    figsize=(14, 7),
    suptitle="🐶🐱 Sample Images for Lab 5",
)

In [ ]:
from diffusers import AutoPipelineForText2Image

# Fast default for practice. Good for short classroom experiments.
T2I_MODEL_ID = "stabilityai/sd-turbo"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"📦 Loading text-to-image model: {T2I_MODEL_ID}")
t2i_pipe = AutoPipelineForText2Image.from_pretrained(
    T2I_MODEL_ID,
    torch_dtype=dtype,
    variant="fp16" if torch.cuda.is_available() else None,
)
t2i_pipe = t2i_pipe.to(device)
if torch.cuda.is_available():
    try:
        t2i_pipe.enable_attention_slicing()
    except Exception:
        pass
print("✅ T2I model loaded!")

In [ ]:
def generate_t2i(prompt, negative_prompt=None, seed=0, num_inference_steps=2,
                 guidance_scale=0.0, height=512, width=512, save_name=None):
    generator_device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    start = time.time()
    result = t2i_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        generator=generator,
    )
    image = result.images[0]
    elapsed = time.time() - start
    if save_name:
        save_path = os.path.join(OUT_DIR, save_name)
        image.save(save_path)
        print(f"💾 Saved: {save_path}")
    return image, elapsed

## Demo code

Change the recorded seed. What stays fixed and what changes?

In [ ]:
prompt = "A fluffy white dog as an astronaut on the moon"
images = [generate_t2i(prompt, seed=s, num_inference_steps=2)[0] for s in [0, 1, 2, 3]]
show_image_grid(images, titles=["Seed 0", "Seed 1", "Seed 2", "Seed 3"], cols=4)

In [ ]:
seed_prompt = "A fluffy white dog as an astronaut on the moon, cinematic lighting, realistic photo"
seeds = [0, 1, 2, 3]

seed_images = []
for s in seeds:
    img, _ = generate_t2i(
        seed_prompt,
        seed=s,
        num_inference_steps=2,
        guidance_scale=0.0,
        save_name=f"seed_{s}.png",
    )
    seed_images.append(img)

show_image_grid(
    seed_images,
    titles=[f"Seed = {s}" for s in seeds],
    cols=4,
    figsize=(18, 5),
    suptitle="🎲 Same Prompt, Different Seeds",
)

print(seed_prompt)

## Interpretation

These are actual generated images; the browser displays the measured runs.

Compare the output with the evidence in the input. A generated answer can be wrong.